In [1]:
import torch

self = torch.tensor([
    [1, 1],
    [1, 1],
])
source = torch.tensor([
    [1, 2],
    [3, 4],
])

Given index tensor `[a, b]`, `a` and `b` are indices into the `self` tensor, and the corresponding indices into `source` are the indices of the index tensor itself. To illustrate this more clearly, we can turn the index tensor `[a, b]` into the map:

```
# source index: self index
{
    0: a,
    1: b,
}
```

Let's say `dim=0`. Then the above map indicates that the first row of `source` goes into the a-th row of `self`, and the second row of `source` goes into the b-th row of `self`.

So index `[0, 1]` means that the first row of `source` goes to first row of `self`, and second row to second row:

In [2]:
self.index_reduce(0, torch.tensor([0, 1]), source, 'prod')

/tmp/ipykernel_13892/345027695.py:1: UserWarning: index_reduce() is in beta and the API may change at any time. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1764937973386/work/aten/src/ATen/native/TensorAdvancedIndexing.cpp:1517.)
  self.index_reduce(0, torch.tensor([0, 1]), source, 'prod')


tensor([[1, 2],
        [3, 4]])

Index `[1, 0]` means that the first row of `source` goes to the second row of `self`, and second row to first row:

In [3]:
self.index_reduce(0, torch.tensor([1, 0]), source, 'prod')

tensor([[3, 4],
        [1, 2]])

Index `[1, 1]` means that both rows of `source` go to the second row of `self`.

In [4]:
self.index_reduce(0, torch.tensor([1, 1]), source, 'prod')

tensor([[1, 1],
        [3, 8]])

Index `[0, 0]` means that both rows of `source` go to the first row of `self`.

In [5]:
self.index_reduce(0, torch.tensor([0, 0]), source, 'prod')

tensor([[3, 8],
        [1, 1]])

In general, if `dim=0`, we'll have `self` of size `[M, c1, c2, ..., cn]`, `source` of size `[N, c1, c2, ..., cn]`, and `index` of size `[N]`. The output is the same size as `self`.

In general, `self` has size $[a_0, ..., a_{dim-1}, M, b_0, ..., b_{n}]$, `source` has size $[a_0, ..., a_{dim-1}, N, b_0, ..., b_{n}]$, and `index` has size $[N]$. The output is the same size as `self`.

In [20]:
a = [3, 4, 102, 1]
b = [8, 4, 5]
dim = len(a)
M = 3
N = 10

res = torch.index_reduce(
    torch.randn(a + [M] + b),
    dim,
    torch.zeros([N], dtype=torch.long),
    torch.randn(a + [N] + b),
    'prod',
)

assert res.shape == torch.Size(a + [M] + b)

The proper way to parallelize the operation for MPS is to create a thread for each element of `source`, since every element of `source` goes into one element of the output. Parallelizing with one thread per element of `self` would not be the right way to go since some of the elements of `self` may not even be touched, depending on the values in `index`.

The output buffer will need to be atomic since multiple threads can write to the same element.

In [3]:
import torch

self = torch.tensor([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
])

src = torch.tensor([
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0],
])

self.index_reduce(0, torch.tensor([2, 1]), src, reduce='amin')


RuntimeError: index_reduce_(): Number of indices (2) should be equal to source.size(dim): (3), for dim: 0